# Notebook 05: Inverse Probability Weighting

Applies inverse probability weighting to correct for spatially heterogeneous detection probability and phone penetration rate.

## Setup

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import h3
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

DATA    = Path("../data")
FIGURES = Path("../figures")
FIGURES.mkdir(parents=True, exist_ok=True)
XDR_DIR = DATA / "xdr_exports"
CENSUS  = DATA / "Cartografia_censo2024_R13/Cartografia_censo2024_R13.gpkg"

CRS_GEO = "EPSG:4674"

## Load data

In [ ]:
od = pd.read_parquet(DATA / "od_for_ipw.parquet")
print(f"Trips: {len(od):,} OD pairs, {od['n_trips'].sum():,} total trips")

detection = pd.read_parquet(DATA / "detection_floor.parquet")
print(f"Detection floor: {len(detection):,} sites")

h3_pop = pd.read_parquet(DATA / "h3_population.parquet")
print(f"H3 population: {len(h3_pop):,} cells, "
      f"total pop = {h3_pop['pop_sector'].sum():,.0f}")

tower_act = pd.read_parquet(XDR_DIR / "td_tower_activity")
print(f"Tower activity: {len(tower_act):,} cells")

cell_lookup = pd.read_parquet(DATA / "cell_lookup.parquet")

comunas = gpd.read_file(CENSUS, layer="Comunal_CPV24")

## Phone penetration rate

In [ ]:
cl = cell_lookup.copy()
cl["cell_id"] = cl["cell_id"].astype(int).astype(str)
tower_act["CELL_ID"] = tower_act["CELL_ID"].astype(str)

ta = tower_act.merge(cl[["cell_id", "site_id"]], left_on="CELL_ID", right_on="cell_id", how="inner")
users_per_site = ta.groupby("site_id")["n_users_total"].sum().reset_index()
users_per_site.columns = ["site_id", "observed_users"]

print(f"Sites with activity: {len(users_per_site):,}")

bts = pd.read_parquet(DATA / "bts_sites.parquet")
bts["h3_res8"] = [
    h3.latlng_to_cell(lat, lon, 8) for lat, lon in zip(bts["lat"], bts["lon"])
]
site_h3 = bts[["bts_id", "h3_res8"]].rename(columns={"bts_id": "site_id"})

site_pop = site_h3.merge(
    h3_pop[["h3_index", "pop_sector"]], left_on="h3_res8", right_on="h3_index", how="left",
).drop(columns=["h3_index"])
site_pop["pop_sector"] = site_pop["pop_sector"].fillna(0)

penetration = users_per_site.merge(site_pop, on="site_id", how="left")
penetration["pop_sector"] = penetration["pop_sector"].fillna(0)
penetration["pi"] = np.where(
    penetration["pop_sector"] > 0,
    (penetration["observed_users"] / penetration["pop_sector"]).clip(upper=1.0),
    0.5,
)

print(f"\nPenetration rate:")
print(penetration["pi"].describe().to_string())

## Inclusion probability and IPW weights

In [ ]:
def detection_probability(trip_distance, radius, n_sectors):
    trip_distance = np.asarray(trip_distance, dtype=float)
    radius = np.asarray(radius, dtype=float)
    n_sectors = np.asarray(n_sectors, dtype=float)
    r_eff = np.where(n_sectors > 0, radius / n_sectors, 0.0)
    with np.errstate(divide="ignore", invalid="ignore"):
        result = np.where(r_eff > 0, 1 - np.exp(-trip_distance / r_eff), 1.0)
    return result

od = od.merge(
    penetration[["site_id", "pi"]].rename(columns={"site_id": "site_id_from", "pi": "pi_from"}),
    on="site_id_from", how="left",
)
od["pi_from"] = od["pi_from"].fillna(0.5)

od["delta"] = detection_probability(
    od["displacement_m"].values,
    od["radius_from"].values,
    od["n_sectors_from"].values,
)

od["p_inclusion"] = od["pi_from"] * od["delta"]
od["p_inclusion"] = od["p_inclusion"].clip(lower=0.01)

od["w_raw"] = 1.0 / od["p_inclusion"]
od["w_hajek"] = od["w_raw"] / od["w_raw"].mean()

print("Inclusion probability:")
print(od["p_inclusion"].describe().to_string())
print(f"\nHajek weights:")
print(od["w_hajek"].describe().to_string())

## Aggregate raw and IPW-corrected OD

In [ ]:
od_raw = (
    od.groupby(["codcom_from", "codcom_to"])
    .agg(raw_trips=("n_trips", "sum"))
    .reset_index()
)

od["weighted_trips"] = od["n_trips"] * od["w_hajek"]
od_ipw = (
    od.groupby(["codcom_from", "codcom_to"])
    .agg(ipw_trips=("weighted_trips", "sum"))
    .reset_index()
)

od_compare = od_raw.merge(od_ipw, on=["codcom_from", "codcom_to"])
od_compare["uplift"] = od_compare["ipw_trips"] / od_compare["raw_trips"]

print(f"Comuna OD pairs: {len(od_compare):,}")
print(f"Total raw trips:    {od_compare['raw_trips'].sum():,.0f}")
print(f"Total IPW trips:    {od_compare['ipw_trips'].sum():,.0f}")

## Figure -- IPW uplift map (Fig. 6 in paper)

In [ ]:
uplift_from = (
    od_compare.groupby("codcom_from")
    .agg(raw=("raw_trips", "sum"), ipw=("ipw_trips", "sum"))
    .reset_index()
)
uplift_from["uplift"] = uplift_from["ipw"] / uplift_from["raw"]
uplift_from = uplift_from.rename(columns={"codcom_from": "CUT"})

comunas["CUT"] = comunas["CUT"].astype(int)
comunas_uplift = comunas.merge(uplift_from, on="CUT", how="left")

fig, ax = plt.subplots(figsize=(10, 12))
comunas.boundary.plot(ax=ax, color="#cccccc", linewidth=0.3)
comunas_uplift.dropna(subset=["uplift"]).plot(
    ax=ax, column="uplift", cmap="RdYlGn_r",
    alpha=0.7, edgecolor="grey", linewidth=0.3,
    legend=True, legend_kwds={"label": "IPW uplift (corrected / raw)", "shrink": 0.5},
    vmin=0.8, vmax=2.0,
)
ax.set_title("IPW correction uplift per origin comuna")
ax.set_aspect("equal")
plt.tight_layout()
plt.savefig(FIGURES / "fig_ipw_uplift_map.pdf", bbox_inches="tight", dpi=150)
plt.show()

print("Top 10 comunas by uplift:")
print(uplift_from.nlargest(10, "uplift")[["CUT", "raw", "ipw", "uplift"]].to_string(index=False))

## Save outputs

In [ ]:
od_compare.to_parquet(DATA / "od_comuna_ipw.parquet", index=False)
print(f"Saved: od_comuna_ipw.parquet ({len(od_compare):,} rows)")